## Classes

In [ ]:
# Network
from torch import nn
from typing import Literal

from Lab3.Dropout_regularization import accuracy


class MLPClassifier(nn.Module):
    def __init__(self, n_hidden:int, hidden_dim:int, input_dim:int, output_dim:int, activation:Literal['ReLU', 'Tanh'] = 'ReLU'):
        super(MLPClassifier, self).__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)

        if activation == 'ReLU':
            self.activation = nn.ReLU()
        elif activation == 'Tanh':
            self.activation = nn.Tanh()
        else:
            raise ValueError(f"Unknown activation function: {activation}")

    def forward(self, input):
        x = self.activation(self.input_layer(input))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.output_layer(x)

        return x

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

class MyDataset(Dataset):
    def __init__(self, features_df, target_series):
        """Initializes dataset using pandas DataFrames/Series converted to tensors"""
        self.data = torch.tensor(features_df.values, dtype=torch.float32)
        self.outputs = torch.tensor(target_series.values, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.outputs[idx]


class MyDataModule(pl.LightningDataModule):
    def __init__(self, train_df: pd.DataFrame, test_df: pd.DataFrame, target_name: str, batch_size: int = 32):
        super().__init__()
        self.train_df = train_df
        self.test_df = test_df
        self.target_name = target_name
        self.batch_size = batch_size

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.sampler = None

    def setup(self, stage=None):
        X_test = self.test_df.drop(columns=[self.target_name])
        y_test = self.test_df[self.target_name]
        self.test_dataset = MyDataset(X_test, y_test)

        train_val_df, val_df = train_test_split(self.train_df, test_size=0.2, random_state=42)

        X_train = train_val_df.drop(columns=[self.target_name])
        y_train = train_val_df[self.target_name]

        X_val = val_df.drop(columns=[self.target_name])
        y_val = val_df[self.target_name]

        self.train_dataset = MyDataset(X_train, y_train)
        self.val_dataset = MyDataset(X_val, y_val)

        # --- Setup WeightedRandomSampler for Imbalanced Training Data ---
        class_counts = y_train.value_counts().to_dict()
        class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
        sample_weights = [class_weights[sample] for sample in y_train]

        self.sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, sampler=self.sampler)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)

In [ ]:
from torch import nn

# Lit Model
class LitClassificationModel(pl.LightningModule):
    def __init__(self, n_hidden:int, hidden_dim:int, input_dim:int, output_dim:int, lr:float = 1e-3):
        super(LitClassificationModel, self).__init__()
        self.save_hyperparameters()

        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)
        self.activation = nn.ReLU()

        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input):
        x = self.activation(self.input_layer(input))
        for layer in self.hidden_layers:
            x = self.activation(layer(x))
        x = self.output_layer(x)

        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x).squeeze()
        loss = self.criterion(y_hat, y)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)

        preds = torch.argmax(y_hat, dim=1)
        accuracy = (preds == y).float().mean()

        self.log('val_loss', loss, prog_bar=True)
        self.log('val_accuracy', accuracy, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)

        preds = torch.argmax(y_hat, dim=1)
        accuracy = (preds == y).float().mean()

        self.log('test_loss', loss)
        self.log('test_accuracy', accuracy)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)